# SepsisGuard-UAR — Prepare and train

Select your **A100 / high-RAM runtime**, then **Runtime → Run all**. Approve the private Drive mount when prompted.

The same notebook now continues from your completed preparation checkpoint and trains two predictive candidates: a current-hour logistic baseline and a trailing 12-hour LSTM. It preserves the fitted preprocessing and patient split. A/train supplies optimization and class weights; A/validation selects epochs and the preferred architecture. Calibration and B remain reserved.

Each completed epoch saves model/optimizer/RNG state to private Drive with a verified immutable weight file. A reconnect and Run all resumes at the last completed epoch using the same configuration. The default limits are 8 baseline epochs, 12 LSTM epochs, early stopping after 4 unimproved validation epochs, and a 90-minute training budget per invocation. It tests CUDA bfloat16 for each architecture and falls back to FP32 if the operator is unsupported. CPU training is available through the explicit command-line option.

Outputs are under `MyDrive/SepsisGuard-UAR/training/<training-id>`: selected weights, epoch history, validation metrics and a final `TRAINING_RESULT.json`. Scores are uncalibrated; validation metrics guide selection and are not independent test performance or clinical validation.

Your preparation run `pipeline-20261104T190221Z-22906fa2` is restored automatically. If no completed preparation is available, the notebook performs preparation once before training. Data archives are restored after a runtime reset as needed. No switching between notebooks is required.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, uuid
from google.colab import drive

REPO_URL = 'https://github.com/shakiful/Sepsisguard-Uar.git'
PROJECT = Path('/content/sepsisguard-uar-unified')
# Keep previous notebooks and downloaded records intact.
if PROJECT.exists():
    assert (PROJECT/'.git').is_dir(), 'The chosen source folder is not a Git checkout.'
    remote = subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    assert remote.lower().rstrip('/').removesuffix('.git') == REPO_URL.lower().removesuffix('.git'), 'Unexpected source repository.'
    dirty = subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain','--untracked-files=no'],text=True).strip()
    if dirty:
        PROJECT = Path('/content/sepsisguard-uar-unified-'+uuid.uuid4().hex[:8])
if not PROJECT.exists():
    subprocess.run(['git','clone','--branch','main','--single-branch',REPO_URL,str(PROJECT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(PROJECT),'merge','--ff-only','origin/main'],check=True)
os.chdir(PROJECT)
compute = json.loads(subprocess.check_output([sys.executable,'scripts/compute_check.py'],text=True))
print(json.dumps(compute,indent=2))
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/SepsisGuard-UAR')
DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev,download,torch]'],check=True)
old_data = Path('/content/sepsisguard-uar/data/raw')
DATA_ROOT = old_data if old_data.is_dir() else PROJECT/'data/raw'
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Persistent project folder:',DRIVE_ROOT)


In [ ]:
PREPARED_RUN_ID = 'pipeline-20261104T190221Z-22906fa2'
# These bounded defaults can be reviewed here before starting the A100 run.
BASELINE_EPOCHS = 8
LSTM_EPOCHS = 12
MAX_TRAINING_MINUTES = 90
command = [sys.executable,'-u','-m','scripts.run_training',
           '--data-root',str(DATA_ROOT),'--storage-root',str(DRIVE_ROOT),
           '--prepared-run-id',PREPARED_RUN_ID,'--device','cuda',
           '--baseline-epochs',str(BASELINE_EPOCHS),'--lstm-epochs',str(LSTM_EPOCHS),
           '--max-minutes',str(MAX_TRAINING_MINUTES)]
process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
returncode = process.wait()
if returncode == 2:
    print('Training paused at its time budget. Completed epochs are saved; Run all again resumes with this configuration.')
elif returncode:
    raise RuntimeError('Training stopped. Review the error above; verified checkpoints remain available.')
else:
    print('Predictive training completed. Review TRAINING_RESULT.json and TRAINING_REPORT.md in the private training folder.')
